In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from clustering.clusterize import clusterize
import json
from architecture.ClusterTokenizer import ClusterTokenizer 
from architecture.MultiTabGen import MultiTabGen
from utils.utils import build_cluster_indices
from train.train import train_model
from generate.generate import generate

from torch.utils.data import TensorDataset, DataLoader
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Read modality source and modality target data

In [ ]:
modality_1 = pd.read_parquet("ccle_gene_aligned_final.parquet") #example : transcriptomic data

In [ ]:
modality_2 = pd.read_parquet("ccle_metabo_aligned_final.parquet") #metabolomic data

rows : samples, columns : genes / metabolites

In [5]:
modality_1.shape

(913, 56202)

In [6]:
modality_2.shape

(913, 225)

In [7]:
assert(modality_1.index == modality_2.index).all()

# Clustering of the source modality

The clustering step must be performed beforehand.

Here, we load the clustering results that were generated beforehand and stored in a JSON file. The clustering step is implemented in `clustering/clusterize.py`.

In [ ]:
with open("60_clusters_res_2.4.json", "r", encoding="utf-8") as f:#
    cluster_dict = json.load(f)

`cluster_dict` is a dictionary where each key corresponds to a cluster identifier, and each value contains the list of features assigned to that cluster.
In this example, it containes the list of genes of the sources modality

In [28]:
#cluster_dict

Split into train and test sets

In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(
        modality_1, modality_2, test_size=0.2, random_state=42
    )

Here we build for each cluster, the indices of features it contains

In [11]:
cluster_names, cluster_idx = build_cluster_indices(modality_1.columns, cluster_dict)
print("Using clusters:", len(cluster_names))

Using clusters: 60


## Preprocessing

- If the datasets of modalities are omics data and consists of raw count values, a log-transformation is applied:

\[
\log(x + 1)
\]

- If the data is already log-normalized, only standardization is performed.

- Missing values are imputed.

> In this tutorial, the data is already log-normalized, so the log-transformation step is not applied.

In [12]:
X_train_np = X_train.values if hasattr(X_train, "values") else np.asarray(X_train)
Y_train_np = Y_train.values if hasattr(Y_train, "values") else np.asarray(Y_train)

x_imputer = SimpleImputer(strategy="mean")
x_scaler = StandardScaler()

X_train_imp = x_imputer.fit_transform(X_train_np)
X_train_scaled = x_scaler.fit_transform(X_train_imp)

y_imputer = SimpleImputer(strategy="mean")
y_scaler = StandardScaler()

Y_train_imp = y_imputer.fit_transform(Y_train_np)
Y_train_scaled = y_scaler.fit_transform(Y_train_imp)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train_scaled, dtype=torch.float32)

# Train MultiTabGen

In [13]:
model = MultiTabGen(
    num_features=X_train_tensor.shape[1],
    cluster_idx= cluster_idx,
    m_outputs=Y_train_tensor.shape[1],
    k_latent=5,
    d_model=128,
    hidden_dim=256,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.5
)

In [ ]:
model, history = train_model(
    model=model,
    x_train_tensor=X_train_tensor,
    y_train_tensor=Y_train_tensor,
    num_epochs=150,
    batch_size=32,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True
)

In [15]:
artifacts = {
    "model": model,
    "x_imputer": x_imputer,
    "x_scaler": x_scaler,
    "y_imputer": y_imputer,
    "y_scaler": y_scaler,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
}

# Generate Target Modality

In [16]:
res = generate(X_test, artifacts)

In [17]:
res["s"]

array([[1.0386857, 1.1016167, 1.0712159, ..., 1.1374779, 1.1468208,
        1.1485556],
       [0.9542151, 1.0090021, 0.9723035, ..., 1.0526447, 1.0524492,
        1.0511048],
       [1.1244265, 1.2137634, 1.158741 , ..., 1.232832 , 1.240797 ,
        1.2269691],
       ...,
       [0.9685421, 1.0517834, 1.0263077, ..., 1.0529764, 1.1028758,
        1.0787334],
       [1.0539615, 1.0914587, 1.0829705, ..., 1.1541604, 1.1686367,
        1.1629816],
       [1.0822664, 1.1243676, 1.1213553, ..., 1.1834669, 1.2066959,
        1.217184 ]], dtype=float32)

In [18]:
y_hat = pd.DataFrame(res["y_hat"], columns=Y_test.columns, index=Y_test.index)

In [20]:
y_hat

,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate,aconitate,adenine,adipate,alpha-ketoglutarate,AMP,citrate,...,C56:8 TAG,C56:7 TAG,C56:6 TAG,C56:5 TAG,C56:4 TAG,C56:3 TAG,C56:2 TAG,C58:8 TAG,C58:7 TAG,C58:6 TAG
index,,,,,,,,,,,,,,,,,,,,,
KP4_PANCREAS,6.006918,6.129356,6.054901,5.963136,5.923666,5.922637,5.887938,5.793724,5.854542,5.772085,...,5.717780,5.665796,5.704165,5.620162,5.707965,5.855994,5.855633,5.662732,5.577774,5.612031
TE14_OESOPHAGUS,5.788847,6.014880,5.645859,5.873764,5.933695,6.045579,5.807176,5.821403,5.993248,5.950109,...,5.865412,5.820047,5.694108,5.679945,5.808082,5.836513,5.847486,5.709516,5.652551,5.638851
KLE_ENDOMETRIUM,5.918756,5.818824,5.699305,5.930773,5.924290,5.862251,5.884816,5.995887,5.858349,5.891901,...,5.813903,5.872188,5.805698,5.805392,5.844278,5.776805,5.748644,5.923131,5.917400,5.755299
KMBC2_URINARY_TRACT,5.911200,5.939362,5.646291,5.987693,5.970989,5.977471,5.990498,5.851858,5.855504,5.925908,...,5.873145,5.888363,5.896865,5.910111,5.826115,5.818014,5.765684,5.855861,5.909022,5.813662
KMRC2_KIDNEY,5.805973,5.957391,5.755759,5.968569,5.885393,5.884753,5.927911,6.260421,5.657461,5.829212,...,5.820765,5.763358,5.829368,5.771778,5.748013,5.751088,5.938965,5.767656,5.672225,5.796001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HCC2108_LUNG,5.931134,6.018482,6.074521,5.983729,5.956404,5.699321,5.888791,6.010375,5.727830,5.871033,...,5.907727,5.872321,5.912554,5.883938,5.904811,5.916668,6.100226,5.904711,5.859128,5.856263
SNUC2A_LARGE_INTESTINE,5.824319,5.885888,5.764678,6.007030,5.799446,5.919172,5.842176,5.786239,6.043603,5.819109,...,5.835145,5.907181,5.884633,5.709386,5.662865,5.710987,5.754923,5.817446,5.699175,5.719087
NCIH596_LUNG,6.012520,5.909610,5.411682,6.088670,5.955971,5.807981,5.963966,5.888248,5.943602,5.913077,...,5.963903,5.941185,5.942816,5.886213,5.740086,5.719835,5.676867,5.928218,5.919849,5.810359


In [19]:
Y_test

,2-aminoadipate,3-phosphoglycerate,alpha-glycerophosphate,4-pyridoxate,aconitate,adenine,adipate,alpha-ketoglutarate,AMP,citrate,...,C56:8 TAG,C56:7 TAG,C56:6 TAG,C56:5 TAG,C56:4 TAG,C56:3 TAG,C56:2 TAG,C58:8 TAG,C58:7 TAG,C58:6 TAG
index,,,,,,,,,,,,,,,,,,,,,
KP4_PANCREAS,5.563285,6.448377,6.220426,6.087582,5.586396,6.848613,5.705635,5.861292,6.121956,5.710258,...,5.316683,5.449888,5.334780,5.325449,5.770549,5.982475,6.427252,5.548336,5.505740,5.225470
TE14_OESOPHAGUS,5.665855,5.753143,5.107697,6.201218,5.989759,6.506930,5.951042,5.830506,5.548577,5.970890,...,5.928705,5.515615,5.906475,5.676064,5.434414,5.530380,5.625645,5.673805,5.503908,5.718855
KLE_ENDOMETRIUM,5.690023,5.458808,5.296257,5.779547,5.986737,5.928369,5.958596,5.908298,5.794343,5.995080,...,5.497806,5.565710,5.632057,5.441338,5.280388,5.496533,5.490214,5.530708,5.675243,5.833841
KMBC2_URINARY_TRACT,5.696266,5.860146,5.880668,6.205871,5.555315,6.241806,5.981897,5.584653,5.860444,5.787430,...,5.807925,6.145312,6.095237,6.026416,5.841738,5.884666,5.650638,6.213003,6.005431,6.129751
KMRC2_KIDNEY,5.999171,5.574656,5.867149,5.708521,5.536090,5.661936,5.740511,6.107778,5.641792,5.490198,...,5.908435,5.459686,6.078518,5.775380,5.893601,5.893449,5.766142,5.850468,5.845083,5.857075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HCC2108_LUNG,5.841102,5.791549,5.987218,5.235099,5.865232,6.239891,5.941624,5.862570,6.322974,5.743494,...,6.098975,6.558055,6.446318,6.682485,6.522184,6.937026,7.177559,6.351620,6.315341,6.655203
SNUC2A_LARGE_INTESTINE,5.636878,5.578258,5.601024,5.670577,5.816466,4.774458,5.835225,5.714172,6.005634,5.829789,...,5.718295,6.012238,5.912423,6.160036,6.037318,6.051057,5.970792,5.851078,5.917349,6.129827
NCIH596_LUNG,6.004470,5.471550,5.837521,5.759753,5.819223,5.749261,5.839720,5.840426,5.721851,5.780503,...,6.003480,6.429164,6.201415,6.314703,6.075611,6.077519,5.606436,6.538741,6.678093,6.330201


# Evaluate

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_metabolites(df_test: pd.DataFrame, df_generated: pd.DataFrame):
    
    assert (df_test.index == df_generated.index).all(), "Index mismatch"
    assert (df_test.columns == df_generated.columns).all(), "Columns mismatch"
    
    y_true = df_test.values
    y_pred = df_generated.values
  
    rmse_global = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_global = mean_absolute_error(y_true, y_pred)
    r2_global = r2_score(y_true, y_pred)
    
    r2_per_met = []
    
    for i, col in enumerate(df_test.columns):
        r2 = r2_score(df_test[col].values, df_generated[col].values)
        r2_per_met.append(r2)
    
    r2_per_met = pd.Series(r2_per_met, index=df_test.columns)
    
    results = {
        "RMSE_global": rmse_global,
        "MAE_global": mae_global,
        "R2_global": r2_global,
        "R2_per_metabolite": r2_per_met
    }
    
    return results

In [22]:
Y_test = pd.DataFrame(
    y_imputer.transform(Y_test.values),
    index=Y_test.index,
    columns=Y_test.columns
)

In [25]:
results = evaluate_metabolites(Y_test, y_hat)

print("RMSE global:", results["RMSE_global"])
print("MAE global:", results["MAE_global"])
print("R2 global:", results["R2_global"])

print("\nTop 10 R2 metabolite:")
print(results["R2_per_metabolite"].sort_values(ascending=False).head(30))

RMSE global: 0.35518570175934333
MAE global: 0.2596217141209578
R2 global: 0.14026691099123723

Top 10 R2 metabolite:
1-methylnicotinamide        0.620954
phosphocreatine             0.478707
AMP                         0.386347
C20:4 CE                    0.381455
oxalate                     0.380467
C18:1 CE                    0.377351
UMP                         0.374216
C20:5 CE                    0.374011
CMP                         0.368674
C46:2 TAG                   0.348352
C16:0 CE                    0.344610
N-carbamoyl-beta-alanine    0.343826
C40:6 PC                    0.330632
C22:6 CE                    0.329254
cytidine                    0.329242
C36:4 PC-B                  0.325804
C38:4 PC                    0.314758
C16:1 CE                    0.309085
C46:1 TAG                   0.306946
xanthine                    0.304056
C38:5 PC                    0.301938
C48:2 TAG                   0.291940
C18:2 CE                    0.284078
creatine                    0.2

In [26]:
from scipy.stats import pearsonr

def pearson_summary(Y_true: pd.DataFrame, Y_pred: pd.DataFrame):
    assert (Y_true.index == Y_pred.index).all()
    assert (Y_true.columns == Y_pred.columns).all()

    r_dict = {}
    
    for col in Y_true.columns:
        y = Y_true[col].values
        yhat = Y_pred[col].values
        
        if np.std(y) == 0 or np.std(yhat) == 0:
            r = np.nan
        else:
            r, _ = pearsonr(y, yhat)
        
        r_dict[col] = r

    r_series = pd.Series(r_dict, name="pearson_r")

    mean_r = np.nanmean(r_series)
    median_r = np.nanmedian(r_series)
    pct_above_02 = np.mean(r_series > 0.2) * 100
    pct_above_03 = np.mean(r_series > 0.3) * 100
    pct_above_04 = np.mean(r_series > 0.4) * 100
    pct_above_05 = np.mean(r_series > 0.5) * 100
    pct_above_06 = np.mean(r_series > 0.6) * 100
    pct_above_07 = np.mean(r_series > 0.7) * 100

    print(f"Mean Pearson: {mean_r:.4f}")
    print(f"Median Pearson: {median_r:.4f}")
    print(f"% metabolites with r > 0.2: {pct_above_02:.2f}%")
    print(f"% metabolites with r > 0.3: {pct_above_03:.2f}%")
    print(f"% metabolites with r > 0.4: {pct_above_04:.2f}%")
    print(f"% metabolites with r > 0.5: {pct_above_05:.2f}%")
    print(f"% metabolites with r > 0.6: {pct_above_06:.2f}%")
    print(f"% metabolites with r > 0.7: {pct_above_07:.2f}%")

    return r_series

In [27]:
r_met = pearson_summary(Y_test, y_hat)

Mean Pearson: 0.3901
Median Pearson: 0.3976
% metabolites with r > 0.2: 91.56%
% metabolites with r > 0.3: 75.11%
% metabolites with r > 0.4: 49.33%
% metabolites with r > 0.5: 19.11%
% metabolites with r > 0.6: 4.00%
% metabolites with r > 0.7: 0.44%
